In [20]:
"""
INTERACTIVE WORLD MAP (no Mapbox) — with Accessible HTML
=========================================================
- Input data:  data_clean.csv
- Map shapes:  countries.geojson.json  (or countries.geojson)
- Join key   : GeoJSON country *name* (not ISO code) → Kosovo works reliably
- Name match : exact → alias → fuzzy (RapidFuzz if available, else difflib)
- Output     : interactive_country_map.html  (interactive)
               map_accessible.html           (accessible: summary + tables)
               name_match_review.csv         (how each name was matched)
               unmapped_entities.csv         (list of names not matched)

Design notes
------------
• Two choropleth layers: zeros (grey, bottom) + non-zeros (blue, top) → every country visible.
• WG labels:
    - Map hover / dropdown / colorbar: “Working Group (WG) N”
    - Tables: short “wgN” columns (lower-case “wg”)
"""

# ---------- imports & settings ----------
import os, re, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"   # opens the interactive chart in your default browser

# ---------- file paths ----------
CSV_PATH = "data_clean.csv"
GJ_PATH  = "countries.geojson.json" if os.path.exists("countries.geojson.json") else "countries.geojson"

# ============================================================================ #
# 1) LOAD FILES
# ============================================================================ #
df = pd.read_csv(CSV_PATH, low_memory=False)

with open(GJ_PATH, "r", encoding="utf-8") as f:
    gj = json.load(f)

# Pick the best "name-like" property from the GeoJSON (varies by dataset)
prop0 = gj["features"][0]["properties"]
name_key_candidates = ["name", "NAME", "ADMIN", "NAME_EN", "COUNTRY"]
name_key = next((k for k in name_key_candidates if k in prop0), None)
if not name_key:
    raise ValueError(
        "Couldn't find a name-like property in GeoJSON. "
        f"Tried: {', '.join(name_key_candidates)}"
    )

# Set of country names exactly as stored in the GeoJSON (our target vocabulary)
GEO_NAMES = {feat["properties"][name_key] for feat in gj["features"]}
# Case-insensitive helper map (so “france” → “France”)
GEO_NAME_MAP_LC = {n.lower(): n for n in GEO_NAMES}
GEO_NAMES_LIST = sorted(GEO_NAMES)

# ============================================================================ #
# 2) DETECT COLUMNS & LIGHT CLEANING
# ============================================================================ #
# Prefer already-clean column; otherwise use 'country'
country_col = "country_clean" if "country_clean" in df.columns else (
              "country"       if "country"       in df.columns else None)
if not country_col:
    raise ValueError("No 'country_clean' or 'country' column found in data_clean.csv.")

# MC/Core flags (coerce to boolean if needed)
mc_col   = "mc_member"  if "mc_member"  in df.columns else None
core_col = "core_group" if "core_group" in df.columns else None

# Working-group columns: anything starting with 'wg' + a digit (e.g., wg1_... wg2_...)
wg_cols = [c for c in df.columns if re.match(r"(?i)^wg\d", c)]
# Stable numeric order by the first number found in the name
wg_cols = sorted(wg_cols, key=lambda x: int(re.findall(r"\d+", x)[0]) if re.findall(r"\d+", x) else 99)

# Optional broad WG flag (“is a WG member at all”)
wg_member_col = "wg_member" if "wg_member" in df.columns else None

# Light clean: normalize NBSPs and trim whitespace
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.replace("\u00A0", " ", regex=False).str.strip()

def yn_to_bool(s: pd.Series) -> pd.Series:
    """Map common truthy strings to True; everything else → False."""
    return (s.astype(str).str.strip().str.lower()
            .map({"y": True, "yes": True, "member": True, "1": True, "true": True, "t": True, "x": True})
            .fillna(False))

# Coerce MC/Core to booleans (create the column if it’s missing)
if mc_col:
    if df[mc_col].dtype != bool:
        df[mc_col] = yn_to_bool(df[mc_col])
else:
    df["mc_member"] = False
    mc_col = "mc_member"

if core_col:
    if df[core_col].dtype != bool:
        df[core_col] = yn_to_bool(df[core_col])
else:
    df["core_group"] = False
    core_col = "core_group"

# Coerce WG columns to booleans
for c in wg_cols:
    if df[c].dtype != bool:
        df[c] = yn_to_bool(df[c])

# Any_WG = OR of explicit WG columns (+ wg_member if present)
df["any_wg"] = df[wg_cols].any(axis=1) if wg_cols else False
if wg_member_col:
    df["any_wg"] = df["any_wg"] | yn_to_bool(df[wg_member_col])

# ============================================================================ #
# 3) NAME MATCHING: exact → alias → fuzzy  (ALIAS FIRST for non-exact, as requested)
# ============================================================================ #
# Try RapidFuzz for stronger fuzzy matching; fall back to difflib if not installed.
try:
    from rapidfuzz import process as rf_process, fuzz as rf_fuzz
    HAVE_RF = True
except Exception:
    from difflib import get_close_matches, SequenceMatcher
    HAVE_RF = False

# Alias map: **CSV → GeoJSON**. Keep this short; fuzzy will handle the rest.
# (Examples reflect names you showed in your GeoJSON.)
ALIASES_TO_GJ = {
    # Kosovo passes as-is
    "kosovo*": "Kosovo", "kosovo": "Kosovo", "republic of kosovo": "Kosovo",

    # Examples from your GeoJSON naming
    "czech republic": "Czechia",
    "macedonia": "North Macedonia",
    "serbia": "Republic of Serbia",
    "united states": "United States of America",
    "usa": "United States of America",

    # Common modern/legacy (lowercased keys for robustness)
    "uk": "United Kingdom",
    "türkiye": "Turkey", "turkiye": "Turkey", "turkey": "Turkey",
    "côte d’ivoire": "Ivory Coast", "cote d'ivoire": "Ivory Coast",
}

# Non-country groups we don’t want to map onto the choropleth
STOPLIST = {
    "European Commission and EU Agencies",
    "European RTD Organisations",
    "European Commission",
    "European Union", "EU",
}

def normalize_name(s: str) -> str:
    """Basic cleanup before alias/fuzzy:
    - drop trailing parentheses like 'Türkiye (TR)'
    - normalize curly apostrophe to straight
    - collapse internal whitespace
    - lower-case for alias lookup
    """
    s = (str(s) or "").strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)  # drop trailing " (..)" chunks
    s = s.replace("’", "'")
    s = " ".join(s.split())
    return s.lower()

def fuzzy_best(name_clean: str):
    """Return (best_candidate, score in 0..1) from GeoJSON names using clean input."""
    # Convert back to a readable candidate list; matching is against canonical case
    if HAVE_RF:
        match = rf_process.extractOne(name_clean, GEO_NAMES_LIST, scorer=rf_fuzz.WRatio)
        if match is None:
            return None, 0.0
        cand, score, _ = match
        return cand, score / 100.0
    # difflib fallback
    cand = get_close_matches(name_clean, GEO_NAMES_LIST, n=1, cutoff=0.0)
    if not cand:
        return None, 0.0
    score = SequenceMatcher(None, name_clean, cand[0]).ratio()
    return cand[0], score

FUZZY_THRESHOLD = 0.85  # tweakable balance of safety vs convenience
_cache = {}

def to_geojson_name(raw: str):
    """
    Map a CSV country to a GeoJSON country *name* or None if not mappable.
    Order: exact (case-insensitive) → alias → fuzzy (if score ≥ threshold).
    """
    if pd.isna(raw):
        return None
    if raw in _cache:
        return _cache[raw]

    raw_str = str(raw).strip()
    if raw_str in STOPLIST:
        _cache[raw] = None
        return None

    nrm = normalize_name(raw_str)

    # 1) exact (case-insensitive)
    if nrm in GEO_NAME_MAP_LC:
        _cache[raw] = GEO_NAME_MAP_LC[nrm]
        return _cache[raw]

    # 2) alias (keys are lowercase)
    if nrm in ALIASES_TO_GJ:
        _cache[raw] = ALIASES_TO_GJ[nrm]
        return _cache[raw]

    # 3) fuzzy (only for those not matched by exact/alias)
    cand, score = fuzzy_best(nrm)
    if cand and score >= FUZZY_THRESHOLD:
        _cache[raw] = cand
        return cand

    _cache[raw] = None
    return None

# Apply mapping and write a review CSV so newcomers can see what happened
df["gj_name"] = df[country_col].apply(to_geojson_name)

unique_raw = (df[country_col].dropna()
              .map(lambda x: str(x).strip())
              .drop_duplicates()
              .sort_values())
rows = []
for raw in unique_raw:
    raw_str = str(raw).strip()
    nrm = normalize_name(raw_str)
    exact_ci = GEO_NAME_MAP_LC.get(nrm)
    alias    = ALIASES_TO_GJ.get(nrm)
    cand, score = fuzzy_best(nrm)
    chosen   = to_geojson_name(raw_str)
    method   = ("exact" if exact_ci and chosen == exact_ci else
                "alias" if alias and chosen == alias else
                "fuzzy" if cand and chosen == cand else
                "none")
    rows.append({
        "raw": raw_str,
        "normalized": nrm,
        "exact_match_ci": exact_ci,
        "alias_used": alias,
        "fuzzy_candidate": cand,
        "fuzzy_score": round(score, 3),
        "chosen": chosen,
        "method": method
    })
pd.DataFrame(rows).to_csv("name_match_review.csv", index=False)
print("Wrote: name_match_review.csv")

# ============================================================================ #
# 4) AGGREGATE TO COUNTRY LEVEL
# ============================================================================ #
agg_parts = {
    "gj_name":      ("gj_name", "first"),
    "total_people": (country_col, "size"),
    "MC":           (mc_col, "sum"),
    "Core":         (core_col, "sum"),
    "Any_WG":       ("any_wg", "sum"),
}
for c in wg_cols:
    agg_parts[c] = (c, "sum")

agg = df.groupby("gj_name", dropna=False, as_index=False).agg(**agg_parts)

count_cols = ["total_people", "MC", "Core", "Any_WG"] + wg_cols
for c in count_cols:
    agg[c] = agg[c].fillna(0).astype(int)

# Anything we could not map? Save for fixing/education.
not_in_geojson = (agg.loc[agg["gj_name"].isna() | ~agg["gj_name"].isin(GEO_NAMES), "gj_name"]
                    .dropna().sort_values().unique().tolist())
if not_in_geojson:
    pd.DataFrame({"not_in_geojson": not_in_geojson}).to_csv("unmapped_entities.csv", index=False)
    print("Some names aren't in the GeoJSON. Wrote unmapped_entities.csv")

# Keep only mappable rows for the choropleth
agg_map = agg[agg["gj_name"].isin(GEO_NAMES)].copy()

# ============================================================================ #
# 5) BUILD PLOT STATE (what each dropdown option should show)
# ============================================================================ #
# Map labels (for dropdown/colorbar/hover): “Working Group (WG) N”
map_labels = {"total_people": "Total people",
              "MC": "Management Committee",
              "Core": "Core Group",
              "Any_WG": "Any Working Group"}
for c in wg_cols:
    m = re.search(r"\d+", c)
    n = m.group(0) if m else c
    map_labels[c] = f"Working Group (WG) {n}"

# Table labels (keep 'wg' on tables → short lowercase wg1, wg2, …)
table_labels = {"gj_name": "Country", "total_people": "Total", "MC": "MC", "Core": "Core", "Any_WG": "Any WG"}
for c in wg_cols:
    m = re.search(r"\d+", c)
    n = m.group(0) if m else c
    table_labels[c] = f"wg{n}"

metrics = list(map_labels.keys())          # stable order for the dropdown
metric_totals = {m: int(agg_map[m].sum()) for m in metrics}

# Hover template (first column in customdata is the country name we show)
custom_cols = ["gj_name"] + count_cols
hover_lines = [
    "<b>%{customdata[0]}</b>",
    "Total: %{customdata[1]}",
    "MC: %{customdata[2]}",
    "Core: %{customdata[3]}",
    "Any WG: %{customdata[4]}",
] + [f"{map_labels[c]}: %{{customdata[{i}]}}" for i, c in enumerate(wg_cols, start=5)]
hovertemplate = "<br>".join(hover_lines) + "<extra></extra>"

def build_state(metric: str):
    """Split countries into zero vs non-zero for a given metric."""
    vals = agg_map.set_index("gj_name")[metric]
    nz = vals[vals > 0]
    z  = vals[vals <= 0]
    base = agg_map.set_index("gj_name", drop=False)
    cd_cols = ["gj_name"] + count_cols
    return dict(
        loc_nz = nz.index.tolist(),
        z_nz   = nz.values,
        cd_nz  = base.loc[nz.index, cd_cols].values,
        loc_z  = z.index.tolist(),
        z_z    = [0] * len(z),
        cd_z   = base.loc[z.index, cd_cols].values,
    )

STATE = {m: build_state(m) for m in metrics}

# Color scale cap (avoid the legend being dominated by a single huge country)
PCT_CAP = 98
caps = {}
for m in metrics:
    z = np.asarray(STATE[m]["z_nz"], dtype=float)
    cap = np.percentile(z, PCT_CAP) if z.size else 1.0
    if not np.isfinite(cap) or cap <= 0:
        cap = max(float(z.max()) if z.size else 1.0, 1.0)
    caps[m] = float(cap)

# ============================================================================ #
# 6) DRAW THE FIGURE (two traces)
# ============================================================================ #
LAND_GRAY  = "#E6E6E6"
HEAT_SCALE = [(0.00,"#9CC0FF"), (0.30,"#6FA5FF"), (0.60,"#3F7EE6"), (0.85,"#1F5FCC"), (1.00,"#0B42C1")]

initial_metric = "Any_WG"   # starting view
s = STATE[initial_metric]

fig = go.Figure()

# Base layer: zero-valued countries (grey)
fig.add_trace(go.Choropleth(
    geojson=gj, featureidkey=f"properties.{name_key}",
    locations=s["loc_z"], z=s["z_z"], text=s["loc_z"], customdata=s["cd_z"],
    hovertemplate=hovertemplate, showscale=False,
    colorscale=[(0, LAND_GRAY), (1, LAND_GRAY)],
    marker_line_color="white", marker_line_width=0.5
))

# Top layer: non-zero countries (blue scale)
fig.add_trace(go.Choropleth(
    geojson=gj, featureidkey=f"properties.{name_key}",
    locations=s["loc_nz"], z=s["z_nz"], text=s["loc_nz"], customdata=s["cd_nz"],
    hovertemplate=hovertemplate, colorscale=HEAT_SCALE,
    zmin=0, zmax=caps[initial_metric],
    colorbar=dict(title=map_labels[initial_metric]),
    marker_line_color="white", marker_line_width=0.5
))

TITLE_PREFIX = "EU Network for Evidence-Synthesis in the Agrifood Sector: Members by Country"

# Dropdown to swap the metric: update both traces + title
buttons = []
for m in metrics:
    st = STATE[m]
    buttons.append(dict(
        label=f"{map_labels[m]} ({metric_totals[m]:,})",
        method="update",
        args=[
            {   # per-trace updates
                "locations": [st["loc_z"], st["loc_nz"]],
                "z":         [st["z_z"],   st["z_nz"]],
                "text":      [st["loc_z"], st["loc_nz"]],
                "customdata":[st["cd_z"],  st["cd_nz"]],
                "zmin":      [None, 0],
                "zmax":      [None, caps[m]],
            },
            {   # layout updates
                "title": f"{TITLE_PREFIX} — colouring by {map_labels[m]}",
            }
        ]
    ))

fig.update_layout(
    title=dict(text=f"{TITLE_PREFIX} — colouring by {map_labels[initial_metric]}",
               x=0.5, xanchor="center"),
    margin=dict(l=0, r=0, t=80, b=0),
    updatemenus=[dict(type="dropdown",
                      x=0.99, xanchor="right",
                      y=1.12, yanchor="top",
                      showactive=True,
                      buttons=buttons)],
    geo=dict(
        projection_type="natural earth",
        resolution=110,           # 110 (coarse, default) or 50, 
        showland=True,  landcolor=LAND_GRAY,
        showcountries=True, countrycolor="white",
        showocean=True, oceancolor="white",
        bgcolor="white"
    )
)

fig.show()

# ============================================================================ #
# 7) SAVE INTERACTIVE + ACCESSIBLE PAGE
# ============================================================================ #
# 7a) Interactive file (standalone)
fig.write_html("interactive_country_map.html", include_plotlyjs=True, auto_open=False)
print("Saved: interactive_country_map.html")

# 7b) Accessible page with summary + tables
plot_div = pio.to_html(fig, include_plotlyjs=True, full_html=False)

countries_included   = int(agg_map["gj_name"].nunique())
total_people_all     = int(df.shape[0])
total_people_mapped  = int(agg_map["total_people"].sum())
total_people_unmapped= total_people_all - total_people_mapped
avg_per_country      = round(total_people_mapped / max(1, countries_included), 2)

# Top-5 countries (by total)
top5 = (agg_map[["gj_name","total_people"]]
        .sort_values("total_people", ascending=False)
        .head(5).to_records(index=False))
top5_lines = "".join(f"<li>{name}: {count}</li>" for name, count in top5)

# ----- mapped table (use short 'wgN' in table headers) -----
mapped_cols = ["gj_name", "total_people", "MC", "Core", "Any_WG"] + wg_cols
mapped_table = (agg_map[mapped_cols]
                .rename(columns=table_labels)
                .sort_values("Total", ascending=False))
mapped_html = mapped_table.to_html(index=False, border=0, classes="datatable", table_id="mapped_table")

# ----- NOT ON MAP table with ALL METRICS (groups everything that didn’t match) -----
unmapped_df = df[df["gj_name"].isna()].copy()
if not unmapped_df.empty:
    # Group by a cleaned entity label so "X (EU)" etc. collapse sensibly
    def entity_label(x): return re.sub(r"\s*\([^)]*\)\s*$", "", str(x or "").strip()).replace("’", "'")
    unmapped_df["Entity"] = unmapped_df[country_col].map(entity_label)

    # Base aggregations
    parts = {
        "Entity": ("Entity", "first"),
        "Total": (country_col, "size"),
        "MC": (mc_col, "sum"),
        "Core": (core_col, "sum"),
        "Any WG": ("any_wg", "sum"),
    }

    # Add WG columns using short table headers (wg1, wg2, …)
    wg_table_cols = []
    for c in wg_cols:
        m = re.search(r"\d+", c)  # extract number from e.g. "wg3_develop..."
        if not m:
            continue
        short = f"wg{m.group(0)}"
        parts[short] = (c, "sum")
        wg_table_cols.append(short)

    # Aggregate + type cleanup
    unmapped_agg = unmapped_df.groupby("Entity", as_index=False).agg(**parts).fillna(0)
    for col in ["Total", "MC", "Core", "Any WG"] + wg_table_cols:
        unmapped_agg[col] = unmapped_agg[col].astype(int)

    other_html = (
        "<h2>Not on map (names not matched)</h2>"
        + unmapped_agg.sort_values("Total", ascending=False)
                      .to_html(index=False, border=0, classes="datatable", table_id="unmapped_table")
    )
else:
    other_html = ""

# Page HTML with accessibility-minded structure
page_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Members by Country — Interactive Map</title>
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <style>
    :root {{ --accent: #1F5FCC; }}
    body {{ font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; margin: 1.25rem; line-height: 1.5; color: #222; }}
    .container {{ max-width: 1200px; margin: 0 auto; }}
    .note {{ color: #333; font-size: 0.95rem; }}
    .datatable {{ border-collapse: collapse; width: 100%; margin-top: 1rem; }}
    .datatable th, .datatable td {{ border-bottom: 1px solid #ddd; padding: 0.5rem; text-align: left; }}
    .datatable tr:hover td {{ background: #f6f8fb; }}
    a:focus, button:focus, [tabindex]:focus {{ outline: 3px solid var(--accent); outline-offset: 2px; }}
    .skip-link {{ position: absolute; left: -9999px; top: auto; width: 1px; height: 1px; overflow: hidden; }}
    .skip-link:focus {{ position: static; width: auto; height: auto; padding: .5rem; background: #fff7cc; border: 1px solid #e0c200; }}
    .sr-only {{ position: absolute; width: 1px; height: 1px; padding: 0; margin: -1px; overflow: hidden; clip: rect(0,0,0,0); border: 0; }}
  </style>
</head>
<body>
  <a class="skip-link" href="#tables">Skip to data tables</a>
  <main class="container">
    <section aria-label="Summary of map metrics">
      <h1>EU Network for Evidence-Synthesis in the Agrifood Sector</h1>
      <p>This page shows an interactive world map of members by country. Hover a country to view totals and Working Group counts.
         Use the dropdown above the map to change the metric (Total, MC, Core, WG1…WG5).</p>
      <ul>
        <li><strong>Countries included (mapped):</strong> {countries_included}</li>
        <li><strong>Total people (all rows):</strong> {total_people_all}</li>
        <li><strong>On map (countries only):</strong> {total_people_mapped}</li>
        <li><strong>Not on map (names not matched):</strong> {total_people_unmapped}</li>
        <li><strong>Average per mapped country:</strong> {avg_per_country}</li>
      </ul>
      <details>
        <summary>Top 5 countries by total people</summary>
        <ol>{top5_lines}</ol>
      </details>
      <p class="note">
        Keyboard tip: use Tab to reach the dropdown and plot toolbar (camera icon to download a PNG),
        and arrow keys / +/- to zoom the map. Screen reader users can review the tables below for the same information.
      </p>
    </section>

    <section aria-labelledby="map-heading">
      <h2 id="map-heading">Interactive choropleth map</h2>
      <div role="img"
           aria-label="Choropleth map of members by country. Zero-value countries are shown in light grey; higher values are darker blue. A dropdown lets you switch the metric being mapped.">
        {plot_div}
      </div>
    </section>

    <section id="tables" aria-labelledby="table-heading">
      <h2 id="table-heading">Country counts (mapped)</h2>
      {mapped_html}
      {other_html}
    </section>
  </main>
</body>
</html>"""

with open("map_accessible.html", "w", encoding="utf-8") as f:
    f.write(page_html)
print("Saved: map_accessible.html")


C:\Users\James\AppData\Local\Temp\ipykernel_53184\3876473682.py:87: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Wrote: name_match_review.csv
Saved: interactive_country_map.html
Saved: map_accessible.html
